# Neural Networks I — AI Lab Instructor Notebook

*Deep Learning · Medium*



5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup
*Section: Dataset & Setup*

# Neural Networks I — Student Lab

Focus: perceptron vs MLP, representation power (XOR), and practical MLP training with sklearn.

## Section 0 — Data generators

We’ll use XOR (non-linearly separable) and two-moons (mild nonlinearity).

![XOR dataset (class-colored)](/images/w6-d4-xor-dataset.svg)

![Two-moons dataset (class-colored)](/images/w6-d4-two-moons-dataset.svg)

In [ ]:
$10

**Why this works.** # Neural Networks I — Instructor Lab

Solutions + short rationales. Focus: representation power (XOR), perceptron vs MLP, and practical training with sklearn.

## Section 0 — Data generators

Rationale: XOR is the minimal example proving linear separators are insufficient; two-moons is a realistic nonlinear boundary.

## Task 2: Implement `perceptron_predict(X, w, b)` returning {0,1}.
*Section: Perceptron (linear separator)*

## Section 1 — Perceptron (linear separator)

A perceptron is a linear classifier trained with a mistake-driven update.

### Task 1.1
Implement `perceptron_predict(X, w, b)` returning {0,1}.

### Task 1.2
Implement `train_perceptron(X, y, epochs, lr)` and return `(w, b)`

### Task 1.3
Show it can fit a linearly separable dataset but fails on XOR.

![Linearly separable dataset used in Task 1.3](/images/w6-d4-perceptron-separable-dataset.svg)

### Update Rule Intuition
In `w = w + lr * t * x_i`, the term `t * x_i` is the negative first-order derivative direction (sub-gradient) of the perceptron loss for a misclassified sample, so this is a first-order update step.

In [ ]:
$11

In [ ]:
# Checks
check('perceptron_separable_high_acc', acc_sep > 0.95)
check('perceptron_xor_not_perfect', acc_xor < 0.9)

**Why this works.** ## Section 1 — Perceptron (linear separator)

Rationale: perceptron is a mistake-driven update that converges only for linearly separable data (Perceptron Convergence Theorem). XOR is a counterexample.

## Task 3: Implement `relu(x)`
*Section: MLP forward pass (2-layer)*

## Section 2 — MLP forward pass (2-layer)
We’ll implement only the forward pass. Full backprop/autodiff comes tomorrow.

### Task 2.1
Implement `relu(x)`

ReLU introduces the ability to create curved or piecewise linear decision boundaries.

![ReLU activation function curve](/images/w6-d4-relu-curve.svg)

Reference activation (used heavily in deeper nets):

$$\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}$$

![tanh activation function curve](/images/w6-d4-tanh-curve.svg)

### Task 2.2
Implement `mlp_forward(X, W1, b1, W2, b2)` returning logits (no softmax).

### Task 2.3
Write a function `count_params(d, h, k)` and verify with checks.

In [ ]:
def relu(z: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, z)

def mlp_forward(X: np.ndarray, W1: np.ndarray, b1: np.ndarray, W2: np.ndarray, b2: np.ndarray) -> np.ndarray:
    H = relu(X @ W1 + b1)
    logits = H @ W2 + b2
    return logits

def count_params(d: int, h: int, k: int) -> int:
    return d*h + h + h*k + k

d, h, k = 2, 8, 2
W1 = rng.normal(0, 0.3, size=(d, h))
b1 = np.zeros(h)
W2 = rng.normal(0, 0.3, size=(h, k))
b2 = np.zeros(k)

logits = mlp_forward(X_xor[:5], W1, b1, W2, b2)
check('logits_shape', logits.shape == (5, k))
check('param_count', count_params(2,8,2) == (2*8+8+8*2+2))
print('params(d=2,h=8,k=2)=', count_params(2,8,2))

In [ ]:
# Checks
check('logits_shape', logits.shape == (5, k))
check('param_count', count_params(2,8,2) == (2*8+8+8*2+2))

**Why this works.** ## Section 2 — MLP forward pass (2-layer)

Rationale: adding a hidden layer + nonlinearity creates a universal function approximator (with enough width). Today we only do forward pass + parameter counting.

![ReLU activation function curve](/images/w6-d4-relu-curve.svg)

$$\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}$$

![tanh activation function curve](/images/w6-d4-tanh-curve.svg)

## Task 4: Compare LogisticRegression vs MLP on XOR and two-moons using ROC-AUC (or accuracy if you prefer).
*Section: Train an MLP with sklearn*

## Section 3 — Train an MLP with sklearn

### Task 3.1
Compare LogisticRegression vs MLP on XOR and two-moons using ROC-AUC (or accuracy if you prefer).

HINT: use a Pipeline with StandardScaler.

**Hints**
- use a Pipeline with StandardScaler.

In [ ]:
def eval_models(X, y, name: str):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

    lr = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
    mlp = Pipeline(steps=[('scaler', StandardScaler()), ('model', MLPClassifier(hidden_layer_sizes=(16,16), activation='relu', alpha=1e-4, max_iter=2000, random_state=0))])

    s_lr = cross_validate(lr, X, y, cv=cv, scoring={'acc': 'accuracy'})
    s_mlp = cross_validate(mlp, X, y, cv=cv, scoring={'acc': 'accuracy'})

    print(f"[{name}] LR  acc: {float(np.mean(s_lr['test_acc'])):.3f}")
    print(f"[{name}] MLP acc: {float(np.mean(s_mlp['test_acc'])):.3f}")

eval_models(X_xor, y_xor, 'XOR')
eval_models(X_moons, y_moons, 'moons')

**Why this works.** ## Section 3 — Train an MLP with sklearn

Rationale: a modern interview answer should connect theory to practice: NNs are sensitive to scaling and optimization; pipelines prevent leakage.

## Task 5: Scaling matters
*Section: Failure modes*

## Section 4 — Failure modes

### Task 4.1: Scaling matters
Train an MLP on a *badly scaled* version of the moons dataset and observe performance.

HINT: multiply one feature by 1000 and compare with/without `StandardScaler`.

**Hints**
- multiply one feature by 1000 and compare with/without `StandardScaler`.

In [ ]:
X_bad = X_moons.copy().astype(float)
X_bad[:, 0] = 1000.0 * X_bad[:, 0]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

mlp_no_scale = MLPClassifier(hidden_layer_sizes=(16,16), activation='relu', alpha=1e-4, max_iter=2000, random_state=0)
scores_no = cross_validate(mlp_no_scale, X_bad, y_moons, cv=cv, scoring={'acc': 'accuracy'})

mlp_scaled = Pipeline(steps=[('scaler', StandardScaler()), ('model', MLPClassifier(hidden_layer_sizes=(16,16), activation='relu', alpha=1e-4, max_iter=2000, random_state=0))])
scores_yes = cross_validate(mlp_scaled, X_bad, y_moons, cv=cv, scoring={'acc': 'accuracy'})

print('MLP acc without scaling:', float(np.mean(scores_no['test_acc'])))
print('MLP acc with scaling   :', float(np.mean(scores_yes['test_acc'])))
check('scaling_helps', float(np.mean(scores_yes['test_acc'])) >= float(np.mean(scores_no['test_acc'])) - 1e-6)

In [ ]:
# Checks
check('scaling_helps', float(np.mean(scores_yes['test_acc'])) >= float(np.mean(scores_no['test_acc'])) - 1e-6)

**Why this works.** ## Section 4 — Failure mode: scaling

Rationale: MLP training is gradient-based. Poorly scaled inputs lead to ill-conditioned optimization (tiny/huge gradients), slowing or preventing convergence.